In [ ]:
%load_ext autoreload
%autoreload 2

from math import pi

import matplotlib.pyplot as plt
import numpy as np
import tqdm.contrib.itertools
import xarray as xr

from fluxoniumcr import DATA_DIR
from fluxoniumcr.qubits.fluxonium import Fluxonium
from fluxoniumcr.qubits.product_basis import DressedProductBasis, ProductBasis

In [ ]:
def create_fluxonium(
        EC: float,
        EJdivEC: float,
        ELdivEJ: float,
) -> Fluxonium:
    fx = Fluxonium(
        EJ=EJdivEC * EC,
        EC=EC,
        EL=ELdivEJ*EJdivEC * EC,
        flux=0.5,
        dim=6,
        cutoff=128,
    )
    return fx


def calculate_residual_zz(q1, q2, JC, scale1=1.0, scale2=1.0):
    bare_basis = ProductBasis([q1, q2])
    
    H1 = bare_basis.get_operator({0: 'hamiltonian'})
    H2 = bare_basis.get_operator({1: 'hamiltonian'})
    n1n2 = bare_basis.get_operator(['charge', 'charge'])

    dressed_basis = DressedProductBasis(
        scale1 * H1 + scale2 * H2 + JC * n1n2,
        bare_basis,
        truncated_dims=(2, 2),
    )
    
    evals = dressed_basis.eigenvalues
    residual_zz = evals[3] + evals[0] - (evals[1] + evals[2])

    return residual_zz

# Both $E_J/E_C$ sweep

In [ ]:
target_zz = 50e-6 * (2*pi)
JC_guess = 0.1 * (2*pi)

control_frequency = 0.6 * (2*pi)
target_frequency = 0.7 * (2*pi)

ELdivEJ_1 = 0.25
ELdivEJ_2 = 0.25

dataset = xr.Dataset(
    attrs=dict(
        target_zz=target_zz,
        wc10=control_frequency,
        wt10=target_frequency,
    )
)

ELdivEJ_1_coord = xr.DataArray(
    [ELdivEJ_1],
    dims=['ELdivEJ_1'],
    attrs=dict(
        units="Grad/s"
    ),
)
ELdivEJ_2_coord = xr.DataArray(
    [ELdivEJ_2],
    dims=['ELdivEJ_2'],
    attrs=dict(
        units="Grad/s"
    ),
)

EJdivEC_1_coord = xr.DataArray(
    np.linspace(1, 8, 71),
    dims=['EJdivEC_1'],
    attrs=dict(
        units="Grad/s"
    ),
)

EJdivEC_2_data = xr.DataArray(
    np.linspace(1, 8, 71),
    dims=['EJdivEC_2'],
    attrs=dict(
        units="Grad/s"
    ),
)

dataset['residual_zz'] = xr.DataArray(
    np.nan,
    coords=[ELdivEJ_1_coord, ELdivEJ_2_coord, EJdivEC_1_coord, EJdivEC_2_data],
    attrs=dict(
        units="Grad/s"
    ),
)
dataset['JC'] = xr.DataArray(
    np.nan,
    coords=[ELdivEJ_1_coord, ELdivEJ_2_coord, EJdivEC_1_coord, EJdivEC_2_data],
    attrs=dict(
        units="Grad/s"
    ),
)
dataset['control_qubit_charge_dipole'] = xr.DataArray(
    np.nan,
    coords=[ELdivEJ_1_coord, ELdivEJ_2_coord, EJdivEC_1_coord, EJdivEC_2_data],
)
dataset['target_qubit_charge_dipole'] = xr.DataArray(
    np.nan,
    coords=[ELdivEJ_1_coord, ELdivEJ_2_coord, EJdivEC_1_coord, EJdivEC_2_data],
)


for (
        ELdivEJ_1,
        ELdivEJ_2,
        EJdivEC_1,
        EJdivEC_2,
) in tqdm.contrib.itertools.product(
        dataset.ELdivEJ_1.data,
        dataset.ELdivEJ_2.data,
        dataset.EJdivEC_1.data,
        dataset.EJdivEC_2.data,
):
    ds = dataset.loc[dict(
        ELdivEJ_1=ELdivEJ_1,
        ELdivEJ_2=ELdivEJ_2,
        EJdivEC_1=EJdivEC_1,
        EJdivEC_2=EJdivEC_2,
    )]
    
    fx1_base = create_fluxonium(1.0, EJdivEC_1, ELdivEJ_1)
    fx2_base = create_fluxonium(1.0, EJdivEC_2, ELdivEJ_2)
    scale1  = control_frequency/(fx1_base.eigenvalues[1] - fx1_base.eigenvalues[0])
    scale2  = target_frequency/(fx2_base.eigenvalues[1] - fx2_base.eigenvalues[0])

    JC = JC_guess
    mu_zz = calculate_residual_zz(fx1_base, fx2_base, JC, scale1=scale1, scale2=scale2)
    for _ in range(3):
        JC = JC * np.sqrt(target_zz/abs(mu_zz))
        mu_zz = calculate_residual_zz(fx1_base, fx2_base, JC, scale1=scale1, scale2=scale2)

    ds['residual_zz'][()] = mu_zz
    ds['JC'][()] = JC
    ds['control_qubit_charge_dipole'][()] = abs(fx1_base.get_operator('charge')[0, 1])
    ds['target_qubit_charge_dipole'][()] = abs(fx2_base.get_operator('charge')[0, 1])

In [ ]:
filename_string = (
    f"ELdivEJ_1={ELdivEJ_1},"
    f"ELdivEJ_2={ELdivEJ_2},"
    f"control_frequency={control_frequency/(2*pi)},"
    f"target_frequency={target_frequency/(2*pi)},"
    f"zz_kHz={round(target_zz/(2*pi) * 1e6)}.hdf5"
)
filename_string

In [ ]:
parent_directory = DATA_DIR/"JC_heatmap"
parent_directory.mkdir(parents=True, exist_ok=True)
dataset.to_netcdf(parent_directory/filename_string)